In [21]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("../data/sme_financial_dataset.csv")

# Display first 5 rows
df.head()

,transaction_id,invoice_date,store_id,region,store_type,customer_id,customer_segment,product_id,product_category,quantity,...,net_profit,payment_method,cash_collection_date,expense_payment_date,cash_inflow,cash_outflow,net_cash_flow,is_return,holiday_flag,oil_price_index
0,2316,2022-01-01,14,West,B,705.0,NaN,74,Beverages,2,...,69.23,Card,2022-01-04,2022-01-31,353.60,284.37,69.23,0,0,77.75
1,3674,2022-01-01,21,North,A,7689.0,Wholesale,11,Household,10,...,1803.37,Card,2022-01-01,2022-01-01,6456.71,4653.33,1803.38,0,0,79.72
2,5083,2022-01-01,10,Central,B,7351.0,Regular,38,Clothing,2,...,687.74,Card,2022-01-01,2022-01-01,1663.76,976.02,687.74,0,0,76.06
3,6302,2022-01-01,21,North,A,469.0,Regular,21,Electronics,3,...,481.69,Credit,2022-01-16,2022-01-01,6281.71,5800.02,481.69,0,0,75.09
4,7006,2022-01-01,14,West,B,3525.0,Regular,131,Household,2,...,218.14,UPI,2022-01-01,2022-01-01,818.61,600.46,218.15,0,0,76.31


In [22]:
print("Shape:", df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Shape: (250750, 28)

Data Types:
transaction_id            int64
invoice_date                str
store_id                  int64
region                      str
store_type                  str
customer_id             float64
customer_segment            str
product_id                int64
product_category            str
quantity                  int64
unit_price              float64
promotion                 int64
discount_pct            float64
discount_amount         float64
revenue                 float64
cogs                    float64
gross_profit            float64
operating_expense       float64
net_profit              float64
payment_method              str
cash_collection_date        str
expense_payment_date        str
cash_inflow             float64
cash_outflow            float64
net_cash_flow           float64
is_return                 int64
holiday_flag              int64
oil_price_index         float64
dtype: object

Missing Values:
transaction_id             0
invoice_dat

In [23]:
# Show only columns that contain missing values

missing_values = df.isnull().sum()

print("Columns with missing values:")
print(missing_values[missing_values > 0])

Columns with missing values:
region              2007
customer_id         1001
customer_segment    1502
product_category    1256
discount_pct        1502
payment_method      1002
oil_price_index     1253
dtype: int64


In [24]:
print("Categorical value counts:\n")

print("Region:")
print(df["region"].value_counts(dropna=False))

print("\nCustomer Segment:")
print(df["customer_segment"].value_counts(dropna=False))

print("\nProduct Category:")
print(df["product_category"].value_counts(dropna=False))

print("\nPayment Method:")
print(df["payment_method"].value_counts(dropna=False))

Categorical value counts:

Region:
region
South      50033
Central    49787
West       49724
East       49640
North      49559
NaN         2007
Name: count, dtype: int64

Customer Segment:
customer_segment
Regular      117363
New           69544
Premium       45151
Wholesale     17190
NaN            1502
Name: count, dtype: int64

Product Category:
product_category
Clothing         31392
Pharmacy         31382
Stationery       31355
Grocery          31160
Household        31138
Beverages        31136
Personal Care    31054
Electronics      30877
NaN               1256
Name: count, dtype: int64

Payment Method:
payment_method
UPI              85041
Card             55011
Bank Transfer    44767
Cash             34603
Credit           29827
NaN               1002
upi                167
card               113
bank transfer       95
cash                63
credit              61
Name: count, dtype: int64


In [25]:
# Remove exact duplicate rows

before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Rows before: 250750
Rows after: 250006
Duplicates removed: 744


In [26]:
# Fill missing categorical values with the most frequent value

categorical_columns = [
    "region",
    "customer_segment",
    "product_category",
    "payment_method"
]

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after categorical cleaning:")
print(df[categorical_columns].isnull().sum())

Missing values after categorical cleaning:
region              0
customer_segment    0
product_category    0
payment_method      0
dtype: int64


In [27]:
# Check numerical statistics and possible outliers

numeric_columns = ["discount_pct", "oil_price_index"]

print(df[numeric_columns].describe())

        discount_pct  oil_price_index
count  248506.000000    248756.000000
mean        0.066276        79.008268
std         1.520411         6.835738
min        -5.000000        59.110000
25%         0.015600        73.780000
50%         0.031300        79.010000
75%         0.047000        84.220000
max       105.000000        98.880000


In [28]:
# Check outliers using IQR method

for col in ["discount_pct", "oil_price_index"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    print(f"\n{col}")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Number of outliers:", len(outliers))


discount_pct
Q1: 0.0156
Q3: 0.047
IQR: 0.0314
Lower bound: -0.0315
Upper bound: 0.09409999999999999
Number of outliers: 35739

oil_price_index
Q1: 73.78
Q3: 84.22
IQR: 10.439999999999998
Lower bound: 58.120000000000005
Upper bound: 99.88
Number of outliers: 0


In [29]:
print("Discount percentage values:")
print(df["discount_pct"].sort_values().head(20))

print("\nHighest discount values:")
print(df["discount_pct"].sort_values(ascending=False).head(20))

Discount percentage values:
109559   -5.0
189436   -5.0
110100   -5.0
231773   -5.0
154118   -5.0
236684   -5.0
144475   -5.0
201163   -5.0
74447    -5.0
195448   -5.0
234803   -5.0
985      -5.0
189174   -5.0
121404   -5.0
161749   -5.0
3985     -5.0
241360   -5.0
105290   -5.0
32478    -5.0
99088    -5.0
Name: discount_pct, dtype: float64

Highest discount values:
249903    105.0
564       105.0
243226    105.0
13537     105.0
13730     105.0
66003     105.0
42868     105.0
75998     105.0
40772     105.0
46529     105.0
70705     105.0
74423     105.0
48921     105.0
242385    105.0
77399     105.0
173612    105.0
38194     105.0
144130    105.0
12565     105.0
23944     105.0
Name: discount_pct, dtype: float64


In [30]:
# Handle invalid discount percentages

invalid_discount = (df["discount_pct"] < 0) | (df["discount_pct"] > 100)

print("Invalid discount values:", invalid_discount.sum())

df.loc[invalid_discount, "discount_pct"] = np.nan

print("\nMissing discount values after correction:")
print(df["discount_pct"].isnull().sum())

Invalid discount values: 100

Missing discount values after correction:
1600


In [31]:
# Fill missing discount percentages with the median

discount_median = df["discount_pct"].median()

df["discount_pct"] = df["discount_pct"].fillna(discount_median)

print("Median discount:", discount_median)
print("Missing discount values:", df["discount_pct"].isnull().sum())

Median discount: 0.0313
Missing discount values: 0


In [32]:
# Inspect unit_price for invalid values and outliers

print(df["unit_price"].describe())

print("\nNegative unit prices:", (df["unit_price"] < 0).sum())

print("\nLowest unit prices:")
print(df["unit_price"].sort_values().head(10))

print("\nHighest unit prices:")
print(df["unit_price"].sort_values(ascending=False).head(10))

count    250006.000000
mean        981.310535
std        1401.356478
min       -6878.380000
25%         184.200000
50%         481.050000
75%         929.017500
max       13476.060000
Name: unit_price, dtype: float64

Negative unit prices: 75

Lowest unit prices:
49171    -6878.38
209798   -6691.85
56788    -6354.63
128243   -5610.19
231984   -5290.52
101724   -5140.49
98496    -4847.68
180561   -4833.61
218287   -4691.29
98005    -4455.82
Name: unit_price, dtype: float64

Highest unit prices:
177763    13476.06
67545     12780.90
13582     12511.60
116464    12381.26
11388     11811.18
76229     11342.09
12648     11180.33
113413    11169.94
189624    10958.80
109589    10720.68
Name: unit_price, dtype: float64


In [33]:
# Handle invalid negative unit prices

invalid_price = df["unit_price"] < 0

print("Invalid negative prices:", invalid_price.sum())

df.loc[invalid_price, "unit_price"] = np.nan

print("\nMissing unit prices after correction:")
print(df["unit_price"].isnull().sum())

Invalid negative prices: 75

Missing unit prices after correction:
75


In [34]:
# Fill missing unit prices with the median

price_median = df["unit_price"].median()

df["unit_price"] = df["unit_price"].fillna(price_median)

print("Median unit price:", price_median)
print("Missing unit prices:", df["unit_price"].isnull().sum())

Median unit price: 481.19
Missing unit prices: 0


In [35]:
# Inspect quantity for unusual values

print(df["quantity"].describe())

print("\nMinimum quantity:", df["quantity"].min())
print("Maximum quantity:", df["quantity"].max())

# IQR outlier check
Q1 = df["quantity"].quantile(0.25)
Q3 = df["quantity"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["quantity"] < lower_bound) |
    (df["quantity"] > upper_bound)
]

print("\nIQR lower bound:", lower_bound)
print("IQR upper bound:", upper_bound)
print("Number of quantity outliers:", len(outliers))

count    250006.000000
mean          3.607145
std           2.688453
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max         360.000000
Name: quantity, dtype: float64

Minimum quantity: 1
Maximum quantity: 360

IQR lower bound: -1.0
IQR upper bound: 7.0
Number of quantity outliers: 13027


In [36]:
# Inspect the highest quantity values

print("Quantity frequency for high values:")
print(
    df[df["quantity"] > 10]["quantity"]
    .value_counts()
    .sort_index()
    .tail(30)
)

print("\nRows with very high quantities:")
print(
    df[df["quantity"] > 50][
        ["quantity", "customer_segment", "product_category", "store_type"]
    ].head(20)
)

Quantity frequency for high values:
quantity
11     1937
12     1381
13      817
14      400
15      138
16       50
17       14
18        3
20       15
40       19
60       22
80       22
100      12
120       2
140       2
160       2
180       2
200       1
360       1
Name: count, dtype: int64

Rows with very high quantities:
        quantity customer_segment product_category store_type
15928        140        Wholesale        Beverages          B
18036         60              New    Personal Care          A
20766         80              New         Clothing          C
33005         80              New    Personal Care          D
33838         80          Regular         Pharmacy          C
35916        100              New         Clothing          B
40172        160          Regular        Beverages          B
41206         60          Premium         Pharmacy          B
43281         80          Regular      Electronics          C
44955        360        Wholesale        Househo

In [37]:
# Find a practical upper limit for quantity

for p in [0.95, 0.99, 0.995, 0.999]:
    print(f"{p*100:.1f}th percentile:", df["quantity"].quantile(p))

95.0th percentile: 8.0
99.0th percentile: 12.0
99.5th percentile: 13.0
99.9th percentile: 15.0


In [38]:
# Cap extreme quantity values at the 99.9th percentile

quantity_cap = df["quantity"].quantile(0.999)

print("Quantity cap:", quantity_cap)
print("Values above cap:", (df["quantity"] > quantity_cap).sum())

df["quantity"] = df["quantity"].clip(upper=quantity_cap)

print("\nMaximum quantity after capping:", df["quantity"].max())

Quantity cap: 15.0
Values above cap: 167

Maximum quantity after capping: 15


In [39]:
# Fill missing oil price index values with the median

oil_median = df["oil_price_index"].median()

df["oil_price_index"] = df["oil_price_index"].fillna(oil_median)

print("Median oil price index:", oil_median)
print("Missing oil price index values:", df["oil_price_index"].isnull().sum())

Median oil price index: 79.01
Missing oil price index values: 0


In [40]:
# Inspect date columns before conversion

for col in ["invoice_date", "cash_collection_date", "expense_payment_date"]:
    print(f"\n{col}")
    print(df[col].head(10).to_list())


invoice_date
['2022-01-01', '2022-01-01', '2022-01-01', '2022-01-01', '2022-01-01', '2022-01-01', '2022-01-01', '2022-01-01', '2022-01-01', '2022-01-01']

cash_collection_date
['2022-01-04', '2022-01-01', '2022-01-01', '2022-01-16', '2022-01-01', '2022-01-04', '2022-01-31', '2022-01-01', '2022-01-03', '2022-01-08']

expense_payment_date
['2022-01-31', '2022-01-01', '2022-01-01', '2022-01-01', '2022-01-01', '2022-01-01', '2022-01-08', '2022-01-01', '2022-01-16', '2022-01-01']


In [41]:
# Convert date columns to datetime

date_columns = [
    "invoice_date",
    "cash_collection_date",
    "expense_payment_date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print("Date data types:")
print(df[date_columns].dtypes)

print("\nMissing/invalid dates:")
print(df[date_columns].isnull().sum())

Date data types:
invoice_date            datetime64[us]
cash_collection_date    datetime64[us]
expense_payment_date    datetime64[us]
dtype: object

Missing/invalid dates:
invoice_date            150
cash_collection_date      0
expense_payment_date      0
dtype: int64


In [42]:
# Inspect rows with invalid invoice dates

invalid_dates = df[df["invoice_date"].isna()]

print("Invalid invoice date rows:", len(invalid_dates))

print(
    invalid_dates[
        ["transaction_id", "invoice_date", "cash_collection_date",
         "expense_payment_date", "revenue"]
    ].head(10)
)

Invalid invoice date rows: 150
       transaction_id invoice_date cash_collection_date expense_payment_date  \
891             23220          NaT           2022-01-12           2022-01-05   
1583           216753          NaT           2022-01-07           2022-01-07   
4913           209128          NaT           2022-02-06           2022-01-29   
6908            94724          NaT           2022-02-07           2022-02-07   
7311            34360          NaT           2022-02-02           2022-02-02   
7555            51252          NaT           2022-02-05           2022-02-03   
8342           186107          NaT           2022-02-07           2022-02-06   
8673            32509          NaT           2022-02-08           2022-02-08   
8816           158545          NaT           2022-03-10           2022-02-15   
10787           30857          NaT           2022-02-17           2022-02-17   

       revenue  
891    1072.07  
1583    944.96  
4913    562.52  
6908   2996.79  
731

In [43]:
# Remove rows where invoice_date could not be recovered

before = len(df)

df = df.dropna(subset=["invoice_date"])

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)
print("Missing invoice dates:", df["invoice_date"].isnull().sum())

Rows before: 250006
Rows after: 249856
Rows removed: 150
Missing invoice dates: 0


In [44]:
# Inspect categorical values for inconsistent formatting

print("Payment methods:")
print(df["payment_method"].value_counts(dropna=False))

print("\nStore types:")
print(df["store_type"].value_counts(dropna=False))

Payment methods:
payment_method
UPI              85714
Card             54822
Bank Transfer    44613
Cash             34480
Credit           29728
upi                167
card               113
bank transfer       95
cash                63
credit              61
Name: count, dtype: int64

Store types:
store_type
A      69582
B      60167
C      59913
D      59795
 A       111
 C       101
 D        99
 B        88
Name: count, dtype: int64


In [45]:
# Standardize categorical formatting

# Remove extra spaces
df["payment_method"] = df["payment_method"].str.strip()
df["store_type"] = df["store_type"].str.strip()

# Standardize payment method capitalization
df["payment_method"] = df["payment_method"].str.title()

print("Payment methods after cleaning:")
print(df["payment_method"].value_counts())

print("\nStore types after cleaning:")
print(df["store_type"].value_counts())

Payment methods after cleaning:
payment_method
Upi              85881
Card             54935
Bank Transfer    44708
Cash             34543
Credit           29789
Name: count, dtype: int64

Store types after cleaning:
store_type
A    69693
B    60255
C    60014
D    59894
Name: count, dtype: int64


In [46]:
# Correct UPI capitalization

df["payment_method"] = df["payment_method"].replace("Upi", "UPI")

print("Final payment methods:")
print(df["payment_method"].value_counts())

print("\nFinal store types:")
print(df["store_type"].value_counts())

Final payment methods:
payment_method
UPI              85881
Card             54935
Bank Transfer    44708
Cash             34543
Credit           29789
Name: count, dtype: int64

Final store types:
store_type
A    69693
B    60255
C    60014
D    59894
Name: count, dtype: int64


In [47]:
# Final data-cleaning validation

print("Dataset shape:", df.shape)

print("\nTotal missing values:")
print(df.isnull().sum())

print("\nTotal duplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nInvalid discount values:",
      ((df["discount_pct"] < 0) | (df["discount_pct"] > 100)).sum())

print("Invalid unit prices:",
      (df["unit_price"] < 0).sum())

print("Invalid quantities:",
      (df["quantity"] <= 0).sum())

print("Maximum quantity:", df["quantity"].max())

print("\nDate range:")
print("Start:", df["invoice_date"].min())
print("End:", df["invoice_date"].max())

Dataset shape: (249856, 28)

Total missing values:
transaction_id             0
invoice_date               0
store_id                   0
region                     0
store_type                 0
customer_id             1000
customer_segment           0
product_id                 0
product_category           0
quantity                   0
unit_price                 0
promotion                  0
discount_pct               0
discount_amount            0
revenue                    0
cogs                       0
gross_profit               0
operating_expense          0
net_profit                 0
payment_method             0
cash_collection_date       0
expense_payment_date       0
cash_inflow                0
cash_outflow               0
net_cash_flow              0
is_return                  0
holiday_flag               0
oil_price_index            0
dtype: int64

Total duplicate rows: 6

Data types:
transaction_id                   int64
invoice_date            datetime64[us]
store_id

In [48]:
# Check financial consistency

gross_profit_errors = ~np.isclose(
    df["gross_profit"],
    df["revenue"] - df["cogs"],
    atol=0.01
)

net_profit_errors = ~np.isclose(
    df["net_profit"],
    df["gross_profit"] - df["operating_expense"],
    atol=0.01
)

cash_flow_errors = ~np.isclose(
    df["net_cash_flow"],
    df["cash_inflow"] - df["cash_outflow"],
    atol=0.01
)

print("Gross Profit errors:", gross_profit_errors.sum())
print("Net Profit errors:", net_profit_errors.sum())
print("Cash Flow errors:", cash_flow_errors.sum())

Gross Profit errors: 0
Net Profit errors: 0
Cash Flow errors: 0


In [49]:
# Save cleaned dataset

clean_path = "../data/clean_sme_financial_dataset.csv"

df.to_csv(clean_path, index=False)

print("Clean dataset saved successfully!")
print("Path:", clean_path)
print("Shape:", df.shape)

Clean dataset saved successfully!
Path: ../data/clean_sme_financial_dataset.csv
Shape: (249856, 28)
